In [ ]:
import polars as pl
import statsmodels.api as sm
from typing import Dict

data = pl.read_parquet("../data/1m/1m.parquet").filter(
    (pl.col("symbol") == "ETHUSDT")
    & (pl.col("open_time").is_between(pl.datetime(2023, 1, 1), pl.datetime(2023, 6, 1)))
)


In [ ]:
# kline features
def kline_features() -> Dict[str, pl.Expr]:
    upper_shadow = (pl.col("high") - pl.max_horizontal("open", "close")) / pl.col(
        "open"
    )
    lower_shadow = (pl.min_horizontal("open", "close") - pl.col("low")) / pl.col("open")
    shadow_ratio = (upper_shadow / (upper_shadow + lower_shadow)).fill_nan(0.5)
    length = pl.col("high") - pl.col("low")
    body = (pl.col("close") - pl.col("open")).abs() / length
    body_amp = (pl.col("close") - pl.col("open")).abs() / pl.col("open")
    amp = length / pl.col("open")
    ret = pl.col("close").pct_change()
    direction = (pl.col("close") - pl.col("open")).sign()
    reversion = direction * direction.shift(-1)
    gap = pl.col("open") / pl.col("close").shift(1) - 1
    features = {
        "upper_shadow": upper_shadow,
        "lower_shadow": lower_shadow,
        "shadow_ratio": shadow_ratio,
        "length": length,
        "body": body,
        "body_amp": body_amp,
        "amp": amp,
        "ret": ret,
        "direction": direction,
        "reversion": reversion,
        "gap": gap,
    }
    
    return features

features = kline_features()

lag = 6
data = data.with_columns(
        [
            feat.shift(i).alias(f"{name}_{i}")
            for name, feat in features.items()
            for i in range(lag)
        ]
    ).tail(-lag)


In [ ]:
data.estimated_size(unit="mb")

In [ ]:
data = data.to_pandas()

data[[f"{col}_{i}" for col in features for i in range(6)]].head()

In [ ]:
X = data[[f"{col}_{i}" for col in ["gap"] for i in range(6)]].head(-1)
X = (X - X.mean()) / X.std()
# X = data["inline_ret_1"].to_frame()
y = data["ret_0"].shift(-1).head(-1)

model = sm.OLS(y, sm.add_constant(X))
results = model.fit()
print(results.summary())